# Gold de faturas — V-Energy

Consolida faturamento por **competência**, preserva o detalhe da conta e compara receita entre períodos. Lê apenas Silver Delta. Regras e limites: [dicionário](../../docs/gold/faturas.md).

O padrão `modo=validar` não grava dados. No Job use `modo=publicar`, informe a data de referência do retrato de status e um diretório em Volume para exportação. Execute o notebook inteiro; as fontes são fixadas em versões Delta.

In [ ]:
from pyspark.sql import functions as F
from gold_faturas import (
    COLUNAS_FATURAS,
    COLUNAS_AUDITORIA,
    DIMENSOES,
    validar_data_referencia,
    qualificar,
    validar_destinos,
    ler_publicacao_silver,
    ler_snapshot_delta,
    persistir_quadros,
    construir_faturas,
    validar_reconciliacao,
    deslocar_mes,
    comparar_periodos,
    publicar_revisoes,
    exportar_csv,
)

In [ ]:
from contextlib import ExitStack
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

from gold_contrato_faturas import (
    construir_consumo_contratual,
    segregar_faturas_sem_leitura,
    derivar_complemento,
    cobertura_volume,
    construir_saidas_contratuais,
    exportar_comerciais,
    validar_reconciliacao_contratual,
)

parametros = {
    'tabela_publicacao': 'v-energy.silver.tb_faturas_publicacao',
    'tabela_faturas': 'v-energy.silver.tb_faturas',
    'tabela_pendentes': 'v-energy.silver.tb_faturas_pendentes_cadastro',
    'tabela_cadastro': 'v-energy.silver.tb_unidades_consumidoras',
    'tabela_leituras': 'v-energy.silver.tb_leituras',
    'tabela_gd': 'v-energy.silver.tb_geracao_distribuida',
    'tabela_inspecoes': 'v-energy.silver.tb_inspecoes',
    'tabela_complemento_faturamento': '',
    'tabela_medidores': 'v-energy.silver.tb_medidores',
    'schema_gold': 'v-energy.gold',
    'data_referencia': '',
    'diretorio_exportacao': '',
    'modo': 'validar',
}
for nome, padrao in parametros.items():
    dbutils.widgets.text(nome, padrao)
config = {nome: dbutils.widgets.get(nome).strip() for nome in parametros}
if config['modo'] not in {'validar', 'publicar'}:
    raise ValueError('modo deve ser validar ou publicar')
validar_data_referencia(config['data_referencia'])
schema_gold = qualificar(config['schema_gold'], 2)
validar_destinos(config['schema_gold'], [], [])
if config['modo'] == 'publicar':
    caminho = Path(config['diretorio_exportacao'])
    if not str(caminho).startswith('/Volumes/') or '..' in caminho.parts:
        raise ValueError('Publicação exige diretorio_exportacao em /Volumes/catalogo/schema/volume')

## Leitura consistente e contrato 0.3.0-provisional

A Gold materializa o marcador Silver uma vez e lê suas duas revisões em `versionAsOf`. Não existe fallback para os nomes finais. Cadastro, leituras, GD e inspeções também ficam fixados em snapshots registrados. A Silver de inspeções segue o produtor padronizado no PR #116; confirme sua publicação no workspace.

O complemento é derivado conforme decisão 6587742059: volume analítico ligado à fatura, tarifa base por subgrupo/competência e adicionais fixos do case. Faturas sem leitura são segregadas; falta de doador bloqueia. O snapshot de medidores complementa a evidência de troca pelo mês da retirada. Ver dicionário para proveniência e cobertura.


In [ ]:
def executar_gold():
    batch_id = uuid4().hex
    fontes = {}
    principal, pendentes, publicacao = ler_publicacao_silver(
        spark, config['tabela_publicacao'], config['tabela_faturas'], config['tabela_pendentes']
    )
    fontes.update(publicacao['snapshots'])

    def ler(chave):
        df, fonte = ler_snapshot_delta(spark, config[chave])
        fontes[chave] = fonte
        return df

    cadastro = ler('tabela_cadastro')
    leituras = ler('tabela_leituras')
    gd = ler('tabela_gd')
    inspecoes = ler('tabela_inspecoes')
    if principal.filter(F.col('flag_uc_sem_cadastro')).limit(1).count():
        raise ValueError('A saída principal da Silver contém pendências cadastrais')
    if pendentes.filter(~F.col('flag_uc_sem_cadastro')).limit(1).count():
        raise ValueError('A saída de pendências da Silver contém faturas com cadastro')
    colunas = COLUNAS_FATURAS + [c for c in COLUNAS_AUDITORIA if c in principal.columns]
    fonte_faturas = principal.select(*colunas).unionByName(pendentes.select(*colunas))
    with ExitStack() as recursos:
        recursos.enter_context(persistir_quadros([fonte_faturas]))
        tabelas = construir_faturas(fonte_faturas, cadastro, leituras, config['data_referencia'])
        tabelas = segregar_faturas_sem_leitura(tabelas, leituras)
        recursos.enter_context(persistir_quadros(list(tabelas.values())))
        validar_reconciliacao(fonte_faturas, tabelas)
        if not tabelas['faturas'].limit(1).count():
            raise ValueError('Nenhuma fatura elegível: revise o cadastro Silver')
        comparar_historico(tabelas)
        consumo = construir_consumo_contratual(
            leituras, cadastro, gd, inspecoes, config['data_referencia'], ler('tabela_medidores')
        )
        recursos.enter_context(persistir_quadros([consumo]))
        complemento = (
            ler('tabela_complemento_faturamento')
            if config['tabela_complemento_faturamento']
            else derivar_complemento(consumo, tabelas['faturas'], cadastro)
        )
        recursos.enter_context(persistir_quadros([complemento]))
        contratuais = construir_saidas_contratuais(consumo, tabelas, cadastro, complemento)
        recursos.enter_context(persistir_quadros(list(contratuais.values())))
        validar_reconciliacao_contratual(contratuais, cadastro)
        tabelas['complemento_proveniencia'] = complemento
        tabelas['cobertura_volume'] = cobertura_volume(
            contratuais['uc_consumo_faturamento_mensal'], cadastro
        )
        tabelas['consumo_proveniencia'] = consumo
        tabelas.update(contratuais)
        if config['modo'] == 'validar':
            print('Validação concluída. Nenhum schema, tabela ou CSV foi gravado.')
            return

        def conferir_fontes():
            if any(dbutils.widgets.get(n).strip() != v for n, v in config.items()):
                raise ValueError('Parâmetros mudaram. Execute novamente desde o início.')
            for fonte in fontes.values():
                atual = spark.sql(f'DESCRIBE DETAIL {fonte["tabela"]}').first()
                if atual['id'] != fonte['id_delta']:
                    raise ValueError(f'A identidade da fonte mudou: {fonte["tabela"]}')
            validar_destinos(
                config['schema_gold'],
                [f'{schema_gold}.`tb_gold_faturas_publicacao`'],
                [f['tabela'] for f in fontes.values()] + [config['tabela_publicacao']],
            )

        conferir_fontes()
        limites = consumo.agg(F.min('competencia'), F.max('competencia')).first()
        metadados = {
            'batch_id': batch_id,
            'synthetic': False,
            'reference_date': config['data_referencia'],
            'period': {'start': limites[0].strftime('%Y-%m'), 'end': limites[1].strftime('%Y-%m')},
            'gerado_em_utc': datetime.now(timezone.utc).isoformat(),
            'publicacao_silver': publicacao,
            'fontes': fontes,
        }

        def validar_gravadas(gravadas):
            validar_reconciliacao(fonte_faturas, gravadas)
            validar_reconciliacao_contratual(gravadas, cadastro)

        spark.sql(f'CREATE SCHEMA IF NOT EXISTS {schema_gold}')
        persistidas, versoes = publicar_revisoes(
            spark, tabelas, config['schema_gold'], validar_gravadas, metadados, conferir_fontes
        )
        metadados['versoes_gold'] = versoes
        pasta = Path(config['diretorio_exportacao']) / batch_id
        exportar_csv(persistidas, str(pasta / 'auxiliares'), metadados)
        exportar_comerciais(persistidas, str(pasta / 'comercial'), metadados)
        print(
            f'Revisão Gold confirmada; recibo comercial em {pasta / "comercial"}. '
            'Montar lote de 12 tabelas e executar importador --verificar antes de homologar.'
        )

## Primeiro ano × últimos 12 meses

O intervalo deriva das competências presentes, e cada mês é exigido. Efeitos de preço e volume usam somente leituras utilizáveis e receita pareada. A variação da receita restante aparece separadamente. O preço efetivo inclui tributos, bandeira e mix: não é a tarifa homologada.

In [ ]:
def comparar_historico(tabelas):
    mensal = tabelas['faturamento_mensal']
    limites = mensal.agg(
        F.min('competencia').alias('inicio'),
        F.max('competencia').alias('fim'),
    ).first()
    inicio, fim = limites['inicio'], limites['fim']

    if deslocar_mes(inicio, 23) > fim:
        raise ValueError('A comparação anual exige ao menos 24 meses de histórico')
    tabelas['variacao_receita'] = comparar_periodos(
        mensal,
        inicio.isoformat(),
        deslocar_mes(inicio, 11).isoformat(),
        deslocar_mes(fim, -11).isoformat(),
        fim.isoformat(),
    )
    _validacao_delta = tabelas['variacao_receita'].filter(
        F.col('variacao_receita')
        != F.col('efeito_volume')
        + F.col('efeito_preco_efetivo')
        + F.col('efeito_receita_sem_volume')
    )
    if _validacao_delta.limit(1).count():
        raise ValueError('Decomposição não reconcilia a variação de receita')
    display(tabelas['faturamento_mensal'].orderBy('competencia', *DIMENSOES))
    display(tabelas['variacao_receita'].orderBy('variacao_receita'))
    display(
        tabelas['faturas_excluidas']
        .groupBy('motivo_exclusao')
        .agg(
            F.count('*').alias('faturas'),
            F.sum('valor_total').alias('valor_total'),
        )
    )

## Confirmação persistente, exportação e recuperação

Cada tentativa grava revisões exclusivas e reconcilia seu conteúdo, incluindo `exceptAll` nos dois sentidos. Só depois substitui `tb_gold_faturas_publicacao` numa transação Delta. Consumidores devem materializar esse marcador uma vez e ler `fonte`/`versao_fonte`; os nomes fixos legados não recebem garantia conjunta nem são atualizados por este fluxo.

Após timeout/cancelamento, ler o marcador identifica o conjunto confirmado. Revisões órfãs ficam para manutenção explícita com registro persistente de aposentadoria e retenção de sete dias; UUIDs nunca são reutilizados. Escritor único e ACL são pré-condições. A exportação confere também os arquivos finais contra os DataFrames, publica recibos por último e não fabrica as dez tabelas de outros domínios.


In [ ]:
executar_gold()